# Solutions — React & project setup

Only look here after you've actually tried the exercises in `react_setup.ipynb`.

### LESSON 1 — Exercise

Order matters: `loading` first, then `error`, then the count. Check them in the wrong order
and a failed request that also has `count: 0` reports "No results" instead of the error.

In [ ]:
function l1RenderStatus(state) {
  if (state.loading) return "Loading…";
  if (state.error) return "Something went wrong";
  if (state.count === 0) return "No results";
  return `${state.count} results`;
}

console.log(l1RenderStatus({ loading: true }));
console.log(l1RenderStatus({ loading: false, error: "offline" }));
console.log(l1RenderStatus({ loading: false, error: null, count: 0 }));
console.log(l1RenderStatus({ loading: false, error: null, count: 7 }));

### LESSON 1 — Mini challenge

The original reads and writes `l1Visits`, a variable living outside the function. The state
argument no longer decides the answer on its own, so the same input gives a different
result each time.

React re-runs your UI functions whenever it needs to — sometimes more than once for a
single change. A function whose answer drifts on every call would make the screen show
something different each time React looked at it.

The fix is not to delete the counter. It is to put it in the state, where it is an input
like any other.

In [ ]:
let l1Visits = 0;
function l1RenderGreeting(state) {
  l1Visits += 1;
  return `Hello ${state.name} (visit ${l1Visits})`;
}

const l1Same = { name: "Ada" };
console.log(l1RenderGreeting(l1Same));
console.log(l1RenderGreeting(l1Same));
// Two different answers from one unchanged state — the function is not a function of
// state alone, so React could not rely on it.

In [ ]:
function l1RenderGreetingFixed(state) {
  return `Hello ${state.name} (visit ${state.visit})`;
}

const l1FixedState = { name: "Ada", visit: 1 };
console.log(l1RenderGreetingFixed(l1FixedState));
console.log(l1RenderGreetingFixed(l1FixedState));
console.log(l1RenderGreetingFixed(l1FixedState) === l1RenderGreetingFixed(l1FixedState));

### LESSON 2 — Exercise

This lesson's work happens in the terminal, so the answers are written out rather than run.

1. **Editing the heading.** The page updates without a reload. Vite replaces just the module
   you changed — hot module replacement. This is a dev-server feature, not something the
   browser does on its own.

2. **The two files.**
   - `index.html` contains `<div id="root"></div>` — an empty box, and the only HTML in the
     project.
   - `src/main.jsx` fills it: `createRoot(document.getElementById('root')).render(<App />)`.

   Everything you see on screen is put there by JavaScript.

3. **Inside `dist/`.** An `index.html`, plus an `assets/` folder holding a `.js` and a `.css`
   file with hashes in their names (`index-D_Qsyw3g.js`), plus whatever was in `public/`,
   copied across untouched.

   Your JSX is **gone**. It was compiled into ordinary JavaScript function calls, then
   bundled with React itself and minified. Nothing in `dist/` needs a build step — that is
   the point of a build.

4. **`npm run preview`.** The same page, served from `dist/` instead of from source.

5. **The playground** is installed and running. You will be sent back to it from topic 2
   onwards.

### LESSON 2 — Mini challenge

**Why the source `index.html` fails.** Its script tag is
`<script type="module" src="/src/main.jsx">`. The leading `/` means "the root of the
server" — but over `file://` there is no server, so it resolves to the root of your drive
and the file is never found. The console shows a failed module load; the exact wording
depends on the browser. And even if the file *had* loaded, the browser would have choked on
the JSX inside it, which is not JavaScript it can parse.

**Why the built `dist/index.html` fails.** The JSX is long gone — that file loads
`/assets/index-<hash>.js`, which is ordinary JavaScript. It still fails, and the reason is
the thing both files have in common:

```html
<script type="module" …>
```

**A module script cannot be loaded from the filesystem.** MDN puts it plainly: *"if you try
to load the HTML file locally (i.e., with a `file://` URL), you'll run into CORS errors due
to JavaScript module security requirements. You need to do your testing through a server."*
A `file://` page has no real origin, and module loading refuses it.

So the answer to the challenge is less interesting than it looks: **both pages fail for the
same root reason.** Each also has a second, smaller problem — both script paths start with
`/`, which under `file://` points at the root of your drive rather than at the project — and
the source page has a third, since its module is JSX that no browser can parse. But fixing
the paths would not rescue either page. Nothing loaded from `file://` will.

**What the servers do.** `npm run dev` and `npm run preview` both run a real HTTP server, so
the page has a proper origin and module loading is allowed — and `/src/main.jsx` and
`/assets/...` resolve against the server root instead of your drive. The dev server does one
extra job: it compiles JSX to JavaScript on the fly, as each module is requested. `preview`
has nothing left to compile — `build` already did it.

That is the whole reason a build tool ships with a server command at all.

### LESSON 3 — Exercise

Fix them in the order the browser forces on you.

**Fault 2 — `App` is never exported.** `App.jsx` declares the function and stops there,
while `main.jsx` does `import App from './App.jsx'` — a *default* import with no default
export to bind to.

- **Where you see it:** the **browser console**.

  ```text
  SyntaxError: The requested module '/src/App.jsx' does not provide an export named 'default'
  ```

- **The terminal shows nothing.** This surprises people. `npm run dev` does not bundle your
  app — it serves your files to the browser as native ES modules and transforms each one as
  it is requested. Both files were served successfully, so from the dev server's point of
  view nothing went wrong. It is the *browser* that links the modules together, and the
  browser is where the failure appears.
- **`npm run build` is a different story.** There the bundler does resolve and link
  everything up front, so the same mistake stops the build with a terminal error:

  ```text
  [MISSING_EXPORT] "default" is not exported by "src/App.jsx".
  ```

- Fix: `export default function App() { … }`.

**Fault 1 — the id does not match.** `index.html` declares `<div id="app">`, but `main.jsx`
asks for `getElementById('root')`. That returns `null`, and React refuses it.

- **Where you see it:** the browser console, once the code actually runs.

  ```text
  Error: Target container is not a DOM element.
  ```

- Fix either side, as long as they agree. `root` is the convention.

**Which one first — and why the other hides.** Fault 2 stops the module from ever being
evaluated. `createRoot` is never reached, so React never gets the chance to complain about
the container. Fix the export, reload, and only then does
`Target container is not a DOM element.` appear.

That ordering is the useful lesson. One error at a time is normal: a failure early in the
chain hides everything after it, so fix the first thing the console reports and look again
rather than trying to reason about all of it at once.

### LESSON 3 — Mini challenge

**1. The typo.** The page goes blank and the console shows:

```text
Uncaught Error: Target container is not a DOM element.
```

`getElementById('rooot')` found nothing and returned `null`, so `createRoot` had nothing to
attach to. Recognising this message saves you five minutes every time you meet it.

**2. The paragraph inside the root div.** It vanishes. React clears whatever HTML is inside
the root node the first time `render` runs, because from that moment React manages that
subtree and will not work around content it did not create.

That does *not* make it a bad place for a message — it makes it the standard place for one.
The paragraph is visible while the browser is still downloading and parsing your JavaScript
bundle, and it disappears by itself the instant React takes over. No code needed to remove
it. If you want something on screen before React exists, this is where it goes.